In [28]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np


In [ ]:

df = pd.read_csv("../data/bank-additional-full.csv", sep=";")

In [6]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [7]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [11]:
train_idx = pd.read_csv("../data/splits/v1/train_index.csv")["row_id"]
test_idx = pd.read_csv("../data/splits/v1/test_index.csv")["row_id"]
train = df.loc[train_idx]
test = df.loc[test_idx]

In [12]:
FEATURES = [c for c in df.columns if c not in ("y", "duration")]
X_train, y_train = train[FEATURES], (train["y"] == "yes").astype(int)
X_test, y_test = test[FEATURES], (test["y"] == "yes").astype(int)


In [13]:
FEATURES

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed']

In [14]:
numeric = X_train.select_dtypes(include="number").columns.tolist()
categorical = [c for c in FEATURES if c not in numeric]


In [15]:
numeric


['age',
 'campaign',
 'pdays',
 'previous',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed']

In [18]:
model=Pipeline([
    ('prep',ColumnTransformer([('num',StandardScaler(),numeric),
                                ('cat',OneHotEncoder(handle_unknown='ignore'),categorical),
    ])),
    ('clf',LogisticRegression(max_iter=1000)),
]) 
#numeric, categorical are lists of column names


In [19]:
model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'day_of_week',
                                                   'poutcome'])])),
                ('clf', LogisticRegression(max_iter=1000))])

In [20]:
model.fit(X_train,y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'day_of_week',
                                                   'poutcome'])])),
                ('clf', LogisticRegression(max_iter=1000))])

In [22]:
proba = model.predict_proba(X_test)[:, 1]


In [23]:
print(f"Train rows: {len(train)}, test rows: {len(test)}")
print(f"Test positive rate: {y_test.mean():.1%}")
print(f"AUC:      {roc_auc_score(y_test, proba):.3f}  (honest expectation ~0.79-0.80)")
print(f"Log-loss: {log_loss(y_test, proba):.3f}")
print(f"Brier:    {brier_score_loss(y_test, proba):.3f}")

Train rows: 32950, test rows: 8238
Test positive rate: 30.8%
AUC:      0.748  (honest expectation ~0.79-0.80)
Log-loss: 0.570
Brier:    0.191


In [24]:
print(y_train.mean(), y_test.mean())
print(proba.mean())


0.0637329286798179 0.3083272638990046
0.2572434738908362


In [25]:
(df["y"] == "yes").mean()


0.11265417111780131

In [26]:
print(y_train.mean(), y_test.mean(), proba.mean())


0.0637329286798179 0.3083272638990046 0.2572434738908362


In [34]:
expect_logloss=-(y_test.mean()*np.log(y_test.mean())+(1- y_test.mean())*np.log(1- y_test.mean()))
expct_brier=y_test.mean()*(1-y_test.mean())**2+(1-y_test.mean())*(y_test.mean())**2
print(expect_logloss)
print(expct_brier)

0.6177557289373214
0.21326156223555814
